In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
import resources
from resources.MLdual import DUAL_DATA, DUAL_MODEL, DualLoss, DualStageTransformer
from resources.MLdualHPO import run_dual_hpo
from resources.MLmetrics import postprocess_load_dual_run, postprocess_artifact_table

REPO = Path(resources.__file__).resolve().parents[1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
%load_ext autoreload
%autoreload 2

# Task

Joint UT/FT disorder → two displacement fields → two full curves.

Like CurveOutputs and FieldOutputs, the sections below construct **DAT**, tune it, construct/load **TR_DUAL**, train, save and predict. Detailed plots belong in **ML-DualPostProcessing.ipynb**; study comparisons belong in **ML-HPOpostProcess.ipynb**. No HPC scripts or subprocesses are invoked.

**Existing run:** keep `LOAD_MODEL=True`, `LOAD_DATA=False`; load its checkpoint, metadata and saved predictions without the full dataset. **New run:** set `LOAD_MODEL=False`, `LOAD_DATA=True`, supply `DATA_ROOT`, then enable the training cell. **Fresh HPO:** load DATA and enable only the HPO cell; it uses that same DAT object. Saved results are not a training dataset.

## ML Model Name: Dual Transformer

### DATA

In [ ]:
RUN_ROOT = REPO / "data"
RUN_NAME = "dual-local-trial"
LOAD_RUN = RUN_ROOT / "MULTI/Dual/Transformer/dual-MULTI-trial1"
# An HPO study directory also works: its best/ checkpoint is selected.
LOAD_MODEL = True
LOAD_DATA = False
DATA_ROOT = None  # Parent containing MLdata; set your actual local path.
NSIMS = None      # All paired samples, as in the independent notebooks.
SEED = 42
FIELD_CONFIG = {"components": ("U1", "U2"), "drop_frame0": True, "layout": "auto"}

DAT_DUAL = None
if LOAD_DATA:
    if DATA_ROOT is None:
        raise ValueError("Set DATA_ROOT to the parent containing MLdata.")
    DAT_DUAL = DUAL_DATA.from_files(
        path=DATA_ROOT,
        load=True,
        load_split=False,
        split_frac=0.9,
        split_seed=SEED,
        range_split=(True, False),
        save_split=False,
        LAT="FCC",
        nnx=20,
        dis="disNodes",
        dN=0.2,
        d_data="in",
        mechMode="MULTI",
        nsims=NSIMS,
        model="TR",
        field_input_config=FIELD_CONFIG,
        normalize=True,
    )
DAT = DAT_DUAL
print(f"Using device: {device}")

In [ ]:
if DAT is not None:
    for split in ("train", "val", "test"):
        print(f"{split}: geometry {DAT.splits[split]['geometry'].shape}")
        for mode in ("UT", "FT"):
            print(f"  {mode}: field {DAT.splits[split]['field'][mode].shape}, "
                  f"curve {DAT.splits[split]['curve'][mode].shape}")
elif LOAD_MODEL:
    artifacts, loaded, _, _ = postprocess_load_dual_run(LOAD_RUN)
    if loaded["model"] is None:
        raise FileNotFoundError(f"No dual checkpoint descriptor under {LOAD_RUN}")
    DATA_METADATA = loaded["model"]["data"]
    display(pd.DataFrame(DATA_METADATA["split_sizes"].items(), columns=["Split", "Paired samples"]))
    print("Saved data metadata loaded; DAT is None until raw MLdata is explicitly loaded.")
else:
    print("Set LOAD_DATA=True before constructing a new model or running HPO.")

### HPO

This **runs a fresh study**, rather than loading an existing study's results. The shared `run_dual_hpo` function owns the same conditional architecture, optimizer and loss-weight search used on HPC; ranges are documented in `../HPC/DualOutputs/README.md`. Keep that definition shared rather than maintaining a second search space in this notebook. Edit the budget here. `RESUME_HPO=True` is an explicit study-level resume (interrupted configurations restart from epoch one).

Do not reuse the live HPC study name/directory for a new local experiment. Enable CPU execution only deliberately for small local checks.

In [ ]:
RUN_HPO = False
ALLOW_CPU_HPO = False
STUDY_NAME = "dual-local-hpo"
TARGET_TRIALS = 200
HPO_EPOCHS = 450
TIMEOUT_HOURS = 230
RESUME_HPO = False

if RUN_HPO:
    if DAT is None:
        raise ValueError("Build DAT in the DATA section before running HPO.")
    if device.type != "cuda" and not ALLOW_CPU_HPO:
        raise RuntimeError("CUDA is unavailable; explicitly enable ALLOW_CPU_HPO for a local check.")
    study_DUAL = run_dual_hpo(
        data=DAT,
        study_dir=RUN_ROOT / "MULTI/Dual/Transformer/HPO" / STUDY_NAME,
        study_name=STUDY_NAME,
        target_trials=TARGET_TRIALS,
        epochs=HPO_EPOCHS,
        timeout_hours=TIMEOUT_HOURS,
        seed=SEED,
        device=device,
        resume=RESUME_HPO,
    )
else:
    print("HPO disabled. Set RUN_HPO=True to optimize the DATA built above.")

### MODEL

The explicit defaults below match Trial 1: field encoder 256×4 layers, curve encoder 256×2 layers, four attention heads, normalized MSE and equal outer weights. They are a starting configuration, not an HPO optimum. Edit these dictionaries/constructor arguments as in the independent Outputs notebooks.

Loading reconstructs the saved architecture and loss, ignoring the new-model defaults. With DAT supplied, the loader checks saved sample IDs, normalizers, masks and geometry before allowing predictions. Loading a model is not optimizer-state resume; use a new model/run for fresh training.

In [ ]:
FIELD_MODEL = dict(d_model=256, n_heads=4, n_layers=4, ff_mult=4,
                   dropout=0.20, activation="relu", pos_encoding="learned", norm_first=False)
CURVE_MODEL = dict(d_model=256, n_heads=4, n_layers=2, ff_mult=4,
                   dropout=0.15, activation="relu", pos_encoding="learned", norm_first=False,
                   pool="mean", use_cls_token=True)
LOSS_WEIGHTS = {"field": {"UT": 1.0, "FT": 1.0}, "curve": {"UT": 1.0, "FT": 1.0}}

if LOAD_MODEL:
    artifacts, loaded, DAT, TR_DUAL = postprocess_load_dual_run(
        LOAD_RUN, load_model=True, data=DAT, device=device,
    )
    display(postprocess_artifact_table(artifacts))
else:
    if DAT is None:
        raise ValueError("Build DAT in the DATA section before constructing TR_DUAL.")
    torch.manual_seed(SEED)
    TR_DUAL = DUAL_MODEL(
        model=DualStageTransformer.from_data(DAT, field_kwargs=FIELD_MODEL, curve_kwargs=CURVE_MODEL),
        lossf=DualLoss(curve_loss=nn.MSELoss(), weights=LOSS_WEIGHTS),
        opt=("adamw", 1e-8),
        batch=2,
        lr=1e-4,
        data=DAT,
        scheduler=("plateau", "min", 0.7, 12, 1e-4),
        device=device,
    )

print(TR_DUAL.model)
print(f"Trainable parameters: {sum(p.numel() for p in TR_DUAL.model.parameters() if p.requires_grad):,}")

In [ ]:
RUN_TRAINING = False
N_EPOCHS = 450
EARLY_STOP_PATIENCE = 75
NEW_RUN = RUN_ROOT / "MULTI/Dual/Transformer" / RUN_NAME
RUN_METADATA = {"run_layout": {"task": "MULTI", "output_kind": "Dual",
                               "model": "Transformer", "run_name": RUN_NAME}}

if RUN_TRAINING:
    if LOAD_MODEL or DAT is None:
        raise ValueError("Fresh training requires LOAD_MODEL=False and a loaded DAT.")
    if NEW_RUN.exists():
        raise FileExistsError(f"Choose a new RUN_NAME; refusing to overwrite {NEW_RUN}")
    TR_DUAL.train(
        n_epochs=N_EPOCHS,
        verbose=10,
        early_stop_patience=EARLY_STOP_PATIENCE,
        early_stop_delta=1e-5,
        checkpoint_path=NEW_RUN,
        metadata=RUN_METADATA,
    )

In [ ]:
# Save the best restored weights and all four physical validation outputs.
# No existing checkpoint/results are overwritten in LOAD_MODEL mode.
if RUN_TRAINING:
    TR_DUAL.save(NEW_RUN, metadata=RUN_METADATA)
    DAT.to_json(NEW_RUN / "model_data.json")
    TR_DUAL.save_results(eval_split="val", run_config={"epochs": N_EPOCHS, "seed": SEED}, metadata=RUN_METADATA)
    print("Saved model and validation results:", NEW_RUN)

In [ ]:
# Load saved physical predictions OR explicitly request a new forward pass.
PREDICT_FROM_MODEL = False
PREDICTION_SPLIT = "val"  # Keep the locked test set out of model selection.

if PREDICT_FROM_MODEL:
    if DAT is None:
        raise ValueError("A new forward pass requires the original MLdata loaded as DAT.")
    predictions = TR_DUAL.predict(split=PREDICTION_SPLIT)
    print("New predictions are normalized; DAT.inverse_field/inverse_curve restore physical units.")
else:
    PREDICTIONS_FILE = (artifacts["predictions_npz"] if LOAD_MODEL else NEW_RUN / "results/predictions.npz")
    predictions = {}
    if PREDICTIONS_FILE.is_file():
        with np.load(PREDICTIONS_FILE, allow_pickle=False) as saved:
            predictions = {key: saved[key] for key in saved.files}
        print("Loaded saved physical predictions:", PREDICTIONS_FILE)
    else:
        print("No saved predictions yet. Run training/save, or explicitly enable PREDICT_FROM_MODEL.")
print("Inspect plots in ML-DualPostProcessing.ipynb using this run directory.")